In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 


In [2]:
import pandas as pd
# "C:\Users\fss6k\OneDrive - University of Glasgow\Desktop\GALLANT_WS3_documents_data\CommuniMap\June_25_original_different_formats\data comma sep for GIS and hub.xlsx"
path = "/mnt/c/Users/fss6k/OneDrive - University of Glasgow/Desktop/GALLANT_WS3_documents_data/CommuniMap/June_25_original_different_formats/data comma sep for GIS and hub.xlsx"
# path = "/mnt/c/Users/fss6k/OneDrive - University of Glasgow/Desktop/GALLANT_WS3_documents_data/CommuniMap/June_25_original_different_formats/data comma sep for GIS and hub.csv"
df = pd.read_excel(path)
# df.rename(columns={'Data_Zone':'DataZone'}, inplace=True)
df.head()
# df.describe(include='all')
print(df.columns[:50])


Index(['ID', 'ROOT_ID', 'STATE', 'FEATURED', 'CHECKED', 'VALIDATION_SCORE',
       'FLAG_COUNT', 'LATITUDE', 'LONGITUDE', 'CREATED_AT', 'MODIFIED_AT',
       'USER_ID', 'USER_ROLE', 'CATEGORY', 'PHOTOS', 'TRAVEL_MODE',
       'MOVEMENT_TYPE', 'MOVEMENT_ADDITIONS', 'WHEELING_TYPE',
       'ACTIVITY_SELECTION', 'REDIRECT_TO_ROUTE_RECORDING', 'ROUTE_SELECT',
       'LOG_TYPE', 'CHALLENGE_ENCOUNTERED', 'OTHER_CHALLENGE_ENCOUNTERED',
       'PROBLEM', 'ISSUE_DESCRIPTION', 'POI_TYPE', 'TYPE_TRAVEL_SUPPORT',
       'WEATHER_COMPOST', 'OTHER_WEATHER', 'WEIGHT_OF_FOOD_WASTE',
       'TYPE_OF_WASTE', 'OTHER_TYPE_OF_WASTE', 'FORMAT_CHOICE',
       'VISUAL_AND_PHYSICAL_COMPOST_CONDITION',
       'SOIL_MOISTURE_SENSORY_FEEDBACK', 'OUTSIDE_TEMPERATURE',
       'COMPOST_TEMPERATURE', 'COMPOST_PH_LEVEL',
       'COMPOST_ORGANISMS_AND_ACTIVITY', 'COMPOST_MAINTENANCE_TYPE',
       'OTHER_COMPOST_MAINTANANCE', 'COMPOST_IMPACT',
       'CHALLANGES_AND_BARRIERS', 'COMPOTS_FEELING', 'OTHER_COMPOST_FEELING',

In [20]:
df.iloc[30,111]

'https://dl.spotteron.com/m/s/000073/2025/06/14/1ls1tfls7e66deccwyx76czzsb7vph71'

In [15]:
print(df['Unnamed: 110'][30], df['IMAGE'][30], df['VOICE_DOCUMENT'][30])

KeyError: 'VOICE_DOCUMENT'

In [6]:
print(df.columns[100:111])

Index(['OTHER_WEATHER_GENERAL', 'CAUSE_OF_WEATHER_EVENT',
       'OTHER_CAUSE_WEATHER_EVENT', 'IMPACTS', 'OTHER_WATER_IMPACTS',
       'EMOTION', 'AUDIO_RECORDING', 'DESCRIPTION', 'SPOTTED_AT', 'IMAGE',
       'Unnamed: 110'],
      dtype='object')


In [72]:
# from PIL import Image
# import requests
# from io import BytesIO
# for i in range(100):
#     url = df.IMAGE[i]
#     response = requests.get(url)
#     img = Image.open(BytesIO(response.content))
#     img.save(f"images_GIS_hub/image_{i}.png")
#     img.show()

In [18]:
print(df.iloc[5,:][:60])

ID                                                                                1214167
ROOT_ID                                                                           1214167
STATE                                                                             enabled
FEATURED                                                                                0
CHECKED                                                                                 0
VALIDATION_SCORE                                                                        0
FLAG_COUNT                                                                              0
LATITUDE                                                                        55.800497
LONGITUDE                                                                       -4.330502
CREATED_AT                                                               2025-11-06 16:28
MODIFIED_AT                                                                           NaN
USER_ID   

In [11]:
# pip install folium
import folium
from folium.plugins import MarkerCluster
import pandas as pd

# df already loaded with LATITUDE/LONGITUDE
d = df.copy()
d["LATITUDE"] = pd.to_numeric(d["LATITUDE"], errors="coerce")
d["LONGITUDE"] = pd.to_numeric(d["LONGITUDE"], errors="coerce")
d = d.dropna(subset=["LATITUDE", "LONGITUDE"])

# Scotland rough extent
SCOT_LON_MIN, SCOT_LON_MAX = -8.5, -0.5
SCOT_LAT_MIN, SCOT_LAT_MAX = 54.5, 60.9

# Filter to points in Scotland-ish area (optional)
d = d[
    d["LONGITUDE"].between(SCOT_LON_MIN, SCOT_LON_MAX) &
    d["LATITUDE"].between(SCOT_LAT_MIN, SCOT_LAT_MAX)
]

# Map centered on Scotland
m = folium.Map(location=[56.8, -4.3], zoom_start=6, control_scale=True)

# (Optional) draw a rectangle around Scotland-ish bounding box
folium.Rectangle(
    bounds=[(SCOT_LAT_MIN, SCOT_LON_MIN), (SCOT_LAT_MAX, SCOT_LON_MAX)],
    fill=False, weight=1, tooltip="Scotland extent (approx)"
).add_to(m)

# City overlays: approximate extents with circles (meters)
cities = [
    {"name": "Glasgow",   "lat": 55.8642, "lon": -4.2518, "radius_m": 10000},
    {"name": "Edinburgh", "lat": 55.9533, "lon": -3.1883, "radius_m": 8000},
]
for c in cities:
    folium.Circle(
        location=[c["lat"], c["lon"]],
        radius=c["radius_m"],
        fill=True, fill_opacity=0.05, opacity=0.8,
        tooltip=f'{c["name"]} (~{c["radius_m"]/1000:.0f} km radius)'
    ).add_to(m)
    folium.Marker([c["lat"], c["lon"]], tooltip=c["name"]).add_to(m)

# Add your points (clustered)
mc = MarkerCluster().add_to(m)
for _, row in d.iterrows():
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=3, fill=True, fill_opacity=0.7, opacity=0.7,
    ).add_to(mc)

folium.LayerControl().add_to(m)
m.save("communimap_scotland_with_cities.html")
print("Saved communimap_scotland_with_cities.html")



Saved communimap_scotland_with_cities.html


In [13]:
import requests

url = "https://www.spotteron.com/api/v2/gallery/2092756"
response = requests.get(url)
print(response.status_code)
print(response.headers.get("content-type"))
print(response.text[:1000])  # show first 1000 chars


200
application/json
{"data":{"id":"2092756","type":1,"files":[{"id":2116772,"file":"000073/2025/06/03/k8udhl1a4em8fznwpukcqbytp4dnahgz","state":1,"spec":{"clientUploadId":"baaae519-4c88-490d-b187-3a6d492e0f47","mimeType":"image/jpeg","mediaType":1}}]}}
